<a href="https://colab.research.google.com/github/findsamirks-commits/retail-customer-segmentation-api/blob/main/customer_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛒 Project 3: Retail Customer Segmentation Lab
Welcome to the operations war room! In retail, treating every customer the exact same way wastes money and misses opportunities. Some customers order daily, some only buy during deep discounts, and others are slowly slipping away.

### 🎯 The Business Problem:
* **The Pain:** Marketing teams waste massive budgets sending blanket discounts to everyone, or they fail to notice high-spending customers stopping their orders.
* **The Solution (Unsupervised Learning):** We will use **K-Means Clustering** and **RFM Analysis (Recency, Frequency, Monetary)** to automatically group shoppers into behavioral segments (*e.g., VIP Champions, At-Risk Churners, Bargain Hunters*) so teams can take precise, profitable action!

In [2]:
import pandas as pd
import numpy as np
import datetime as dt

# 1. SIMULATING RETAIL TRANSACTION DATA
# In a real setup, this pulls from your enterprise SQL server (Customer ID, Order Date, Invoice Amount)
np.random.seed(42)
n_transactions = 2000

customer_ids = np.random.randint(1001, 1050, size=n_transactions)
start_date = dt.datetime(2026, 1, 1)
order_dates = [start_date + dt.timedelta(days=int(np.random.randint(0, 90))) for _ in range(n_transactions)]
order_values = np.random.exponential(scale=500, size=n_transactions) + 50 # Grocery/Retail order sizes

df_transactions = pd.DataFrame({
    'CustomerID': customer_ids,
    'OrderDate': order_dates,
    'OrderValue': order_values
})

# 2. CALCULATING RFM METRICS (Recency, Frequency, Monetary)
# Reference date set to the day after our simulation ends
snapshot_date = df_transactions['OrderDate'].max() + dt.timedelta(days=1)

rfm_df = df_transactions.groupby('CustomerID').agg({
    'OrderDate': lambda x: (snapshot_date - x.max()).days, # Recency: How many days since last order?
    'CustomerID': 'count',                                # Frequency: Total number of orders
    'OrderValue': 'sum'                                   # Monetary: Total money spent
}).rename(columns={
    'OrderDate': 'Recency',
    'CustomerID': 'Frequency',
    'OrderValue': 'Monetary'
}).reset_index()

print(f"RFM Dataset successfully built! Total unique customers analyzed: {rfm_df.shape[0]}")
display(rfm_df.head())

RFM Dataset successfully built! Total unique customers analyzed: 49


,CustomerID,Recency,Frequency,Monetary
0,1001,2,40,19549.137024
1,1002,1,42,26549.848615
2,1003,5,43,23346.927736
3,1004,5,35,13847.010041
4,1005,2,42,19414.187744


## 🤖 Step 2: Unsupervised Clustering with K-Means
Unlike supervised learning where we give the AI an "answer key," **Unsupervised Learning** lets the algorithm discover natural groupings on its own.

* **Why we use StandardScaler:** Recency is measured in days (e.g., 5 days), while Monetary is measured in currency (e.g., ₹25,000). If we don't scale them, the large monetary numbers will dominate the math. Scaling gives all three metrics equal weight!

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import joblib
import os

# 1. Select RFM features for the AI
X = rfm_df[['Recency', 'Frequency', 'Monetary']]

# 2. Scale features so they are on the exact same footing
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Initialize and train K-Means (We'll group customers into 4 operational clusters)
# K=4 Clusters: 0=Active Regulars, 1=VIP Champions, 2=At-Risk Churners, 3=Low-Spend Occasional
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm_df['Cluster'] = kmeans.fit_predict(X_scaled)

# 4. Save both the trained clustering model and the scaler to disk
os.makedirs('models', exist_ok=True)
joblib.dump(kmeans, 'models/customer_segment_model.joblib')
joblib.dump(scaler, 'models/rfm_scaler.joblib')

print("K-Means clustering complete and models successfully saved to disk!")
display(rfm_df.head(10))

K-Means clustering complete and models successfully saved to disk!


,CustomerID,Recency,Frequency,Monetary,Cluster
0,1001,2,40,19549.137024,3
1,1002,1,42,26549.848615,0
2,1003,5,43,23346.927736,2
3,1004,5,35,13847.010041,1
4,1005,2,42,19414.187744,3
5,1006,1,40,21210.468178,3
6,1007,5,33,12509.646534,1
7,1008,2,45,25773.983057,0
8,1009,4,39,24192.560603,2
9,1010,4,35,17425.839581,1


## 📦 Step 3: Creating the Internal Employee App Backend (`app.py`)
Now we package our scaler and K-Means model into a high-performance **FastAPI** service. When a category manager or customer service agent enters a customer's shopping metrics, the API instantly outputs their behavior cluster and a recommended operational action.

In [4]:
%%writefile app.py
from fastapi import FastAPI
import joblib
import pandas as pd
from pydantic import BaseModel

# Initialize FastAPI app for internal retail tools
app = FastAPI(title="Retail Customer Segmentation API", version="1.0")

# Load the saved ML model and feature scaler
model = joblib.load('models/customer_segment_model.joblib')
scaler = joblib.load('models/rfm_scaler.joblib')

# Define input structure expected from employees/dashboards
class CustomerInput(BaseModel):
    recency: float      # Days since last purchase
    frequency: float    # Total orders placed
    monetary: float     # Total spend amount

# Dictionary mapping cluster numbers to clear business actions for staff
CLUSTER_MEANINGS = {
    0: {"segment": "Active Regulars", "action": "Maintain routine engagement; recommend cross-category add-ons."},
    1: {"segment": "VIP Champions", "action": "Provide priority delivery, exclusive early access, and loyalty perks."},
    2: {"segment": "At-Risk Churners", "action": "URGENT: Trigger automated WhatsApp retention voucher (15% off) within 24 hours."},
    3: {"segment": "Low-Spend Occasional", "action": "Target with bulk-buy bundling deals to increase basket size."}
}

@app.get("/")
def home():
    return {"message": "Retail Customer Segmentation API is live! Use the /segment endpoint to analyze buyer behavior."}

@app.post("/segment")
def predict_segment(data: CustomerInput):
    # Prepare input data frame
    input_df = pd.DataFrame([{
        'Recency': data.recency,
        'Frequency': data.frequency,
        'Monetary': data.monetary
    }])

    # Scale input using our saved production scaler
    scaled_input = scaler.transform(input_df)

    # Predict cluster
    cluster_id = int(model.predict(scaled_input)[0])
    segment_info = CLUSTER_MEANINGS.get(cluster_id, {"segment": "Unknown", "action": "Review manually."})

    return {
        "input_metrics": data.dict(),
        "assigned_cluster": cluster_id,
        "customer_segment": segment_info["segment"],
        "recommended_employee_action": segment_info["action"]
    }

Writing app.py


In [5]:
# Install required server packages
!pip install -q fastapi uvicorn pyngrok

# Run FastAPI backend in the background
get_ipython().system_raw('uvicorn app:app --host 0.0.0.0 --port 8000 &')

print("Customer Segmentation API is running in the background on port 8000!")
print("Employees or frontend dashboards can now query http://127.0.0.1:8000/docs for instant decision-making.")

Customer Segmentation API is running in the background on port 8000!
Employees or frontend dashboards can now query http://127.0.0.1:8000/docs for instant decision-making.
